In [10]:
import torch

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
import sklearn.linear_model as lm

from tqdm import tqdm

from utils import LogTransformer

seed = 42
np.random.seed(seed)

sns.set_style('darkgrid')
sns.set_theme(font_scale=1.5)


CATEGORICAL_VARIABLES = ["chd", "famhist"]
CONTINUOUS_VARIABLES = ["sbp", "tobacco", "ldl", "typea", "alcohol", "age"]
INCLUDED_VARIABLES = ["sbp", "tobacco", "ldl", "typea", "alcohol", "age", "chd", "famhist"]
LEARNING_RATE = 1e-3
N_EPOCHS = 1000

df = pd.read_csv("data/heartDisease.csv")
df = df.drop(labels=["row.names"], axis=1)


X, y = df[INCLUDED_VARIABLES], df["obesity"]



num_pipeline = Pipeline(steps=[
        ('log_transform', LogTransformer(["alcohol", "tobacco"])),
        ('scaler', StandardScaler().set_output(transform='pandas'))
])


cat_pipeline = Pipeline(steps=[
        ('onehotencoder', OneHotEncoder())
])



preproc = ColumnTransformer([
    ("num", num_pipeline, CONTINUOUS_VARIABLES),
    ("cat", cat_pipeline, CATEGORICAL_VARIABLES),
], remainder='passthrough')

X_preprocessed = preproc.fit_transform(X)


In [49]:
def get_regression_ann(input_dim, hidden_dim):
        return torch.nn.Sequential(
                torch.nn.Linear(in_features=input_dim, out_features=hidden_dim, bias=True),     # Input layer
                torch.nn.Tanh(),                                                                # Activation function
                torch.nn.Linear(in_features=hidden_dim, out_features=1, bias=True)    # Output layer
        )

def get_classification_ann(input_dim, hidden_dim):
        return torch.nn.Sequential(
                torch.nn.Linear(in_features=input_dim, out_features=hidden_dim, bias=True),     # Input layer
                torch.nn.ReLU(),                                                                # Activation function
                torch.nn.Linear(in_features=hidden_dim, out_features=2, bias=True),    # Output layer
                torch.nn.Sigmoid()
        )



In [50]:
hidden_dim = 10

ann = get_regression_ann(X_preprocessed.shape[1], hidden_dim)
losses = []
criterion = torch.nn.MSELoss()
optimizer = torch.optim.SGD(params=ann.parameters(), lr=LEARNING_RATE)


for epoch in range(N_EPOCHS):
    ann.train()
    outputs = ann(torch.tensor(X_preprocessed).float()).reshape(-1)
    loss = criterion(outputs , torch.tensor(y).float())

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()
    
    losses.append(loss.item())



In [51]:
losses

[704.5767211914062,
 698.3043823242188,
 692.0336303710938,
 685.7283325195312,
 679.3544311523438,
 672.8804321289062,
 666.277099609375,
 659.5177612304688,
 652.5783081054688,
 645.4374389648438,
 638.0767211914062,
 630.4807739257812,
 622.6376342773438,
 614.53857421875,
 606.1781005859375,
 597.5542602539062,
 588.6680297851562,
 579.5236206054688,
 570.127685546875,
 560.4896240234375,
 550.6206665039062,
 540.533935546875,
 530.2440185546875,
 519.7669677734375,
 509.1194763183594,
 498.3193664550781,
 487.3852844238281,
 476.3365173339844,
 465.19305419921875,
 453.9755859375,
 442.70538330078125,
 431.4041442871094,
 420.0940246582031,
 408.79742431640625,
 397.536865234375,
 386.3347473144531,
 375.21307373046875,
 364.1933288574219,
 353.2958984375,
 342.5403747558594,
 331.9449462890625,
 321.52618408203125,
 311.2991638183594,
 301.2771301269531,
 291.47161865234375,
 281.89227294921875,
 272.5469055175781,
 263.4417419433594,
 254.58131408691406,
 245.96868896484375,
 23